# Day 2 — Feature Engineering: EDA Notebook

Adaptive Defect Prediction Engine  
Exploratory Data Analysis of the feature matrix produced by `scripts/day2_run.py`.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Environment ready.')

## 1. Load & Inspect Feature Matrix

In [ ]:
FEATURE_MATRIX_PATH = PROJECT_ROOT / 'data' / 'processed' / 'feature_matrix.csv'

df = pd.read_csv(FEATURE_MATRIX_PATH)
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}):')
for col in df.columns:
    print(f'  {col}: {df[col].dtype}')

In [ ]:
# Label distribution
label_counts = df['is_buggy'].value_counts()
print('Label distribution:')
print(label_counts)
print(f'\nBuggy rate: {100 * label_counts.get(1, 0) / len(df):.1f}%')

df.describe()

## 2. Missing Value Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
missing = numeric_df.isnull().mean().sort_values(ascending=False)
missing_nonzero = missing[missing > 0]

if missing_nonzero.empty:
    print('No missing values in numeric columns — imputation worked correctly.')
else:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_nonzero) * 0.4)))
    missing_nonzero.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Missing Value Rate by Feature')
    ax.set_xlabel('Fraction Missing')
    plt.tight_layout()
    plt.show()

## 3. Feature Distributions: Buggy vs Clean

In [ ]:
PROCESS_FEATURES = [
    'code_churn_30d', 'code_churn_90d',
    'commit_count_30d', 'commit_count_90d',
    'author_count_90d', 'bug_fix_count_90d',
    'fix_density', 'days_since_last_change',
    'commit_burst_30d', 'ownership_score',
]

AST_FEATURES = [
    'cyclomatic_complexity', 'cognitive_complexity',
    'max_nesting_depth', 'num_functions', 'num_classes',
    'avg_function_length', 'num_imports', 'ast_node_count',
    'has_try_except', 'max_args_per_function',
]

def plot_feature_distributions(features, title_prefix, df, label_col='is_buggy'):
    available = [f for f in features if f in df.columns]
    n = len(available)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = axes.flatten()

    for i, feat in enumerate(available):
        ax = axes[i]
        for label, color, ls in [(0, 'steelblue', '-'), (1, 'coral', '--')]:
            subset = df[df[label_col] == label][feat].dropna()
            if len(subset) > 1:
                subset.plot.kde(ax=ax, label='Clean' if label == 0 else 'Buggy',
                                color=color, linestyle=ls, bw_method=0.5)
        ax.set_title(feat, fontsize=9)
        ax.legend(fontsize=7)
        ax.set_xlabel('')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'{title_prefix} — Buggy vs Clean KDE', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

plot_feature_distributions(PROCESS_FEATURES, 'Process Features', df)
plot_feature_distributions(AST_FEATURES, 'AST Features', df)

## 4. Correlation Matrix

In [ ]:
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in ('is_buggy',)]

corr = df[feature_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr, mask=mask, annot=False, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax
)
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=12)
plt.tight_layout()
plt.show()

# Highly correlated pairs
pairs = [
    (corr.columns[i], corr.columns[j], corr.iloc[i, j])
    for i in range(len(corr.columns))
    for j in range(i + 1, len(corr.columns))
    if abs(corr.iloc[i, j]) > 0.85
]
if pairs:
    print(f'Highly correlated pairs (|r| > 0.85):')
    for a, b, r in sorted(pairs, key=lambda x: -abs(x[2])):
        print(f'  {a:<35s} {b:<35s} r={r:.3f}')
else:
    print('No feature pairs with |r| > 0.85.')

## 5. Top Features by Mutual Information

In [ ]:
X = df[feature_cols].fillna(0)
y = df['is_buggy'].astype(int)

mi_scores = mutual_info_classif(X, y, discrete_features='auto', random_state=42)
mi_df = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(10, 6))
mi_df.head(top_n).plot(kind='barh', ax=ax, color='darkorange')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Features by Mutual Information with is_buggy')
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print('\nFull MI ranking:')
print(mi_df.to_string())

## 6. Day 2 End-of-Day Summary

In [ ]:
buggy_n = int(df['is_buggy'].sum())
clean_n = int((df['is_buggy'] == 0).sum())
n_features = len(feature_cols)
top3 = mi_df.head(3).index.tolist()

summary = f"""
╔══════════════════════════════════════════════════════════╗
║          Day 2 — Feature Engineering Complete            ║
╠══════════════════════════════════════════════════════════╣
║  Feature matrix shape : {df.shape[0]:>6,} rows × {n_features} features     ║
║  Buggy files          : {buggy_n:>6,}                            ║
║  Clean files          : {clean_n:>6,}                            ║
║  Missing values       : {int(df[feature_cols].isnull().sum().sum()):>6,}                            ║
║  Top MI feature       : {top3[0]:<35s}║
║  2nd MI feature       : {top3[1]:<35s}║
║  3rd MI feature       : {top3[2]:<35s}║
╠══════════════════════════════════════════════════════════╣
║  Next: Day 3 — XGBoost + GNN model training              ║
╚══════════════════════════════════════════════════════════╝
"""
print(summary)